# JSON, Configuration & Environment Variables

## What you'll practice

JSON for data and config; environment variables for deployment-specific settings (12-factor apps, Glue job parameters, Lambda env).
You'll round-trip JSON Lines, overlay env vars onto config with type coercion, validate nested documents, and build a layered settings loader.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/files/test_06_03_json_config_env.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/06_files_json_csv/03_json_config_env.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
INCOMING = DATA / "incoming"
print(sorted(p.name for p in INCOMING.iterdir()))

## Exercise 1 — 🟢 Easy · JSON Lines round trip

**Implement:** `write_jsonl(path, records) -> int`, `read_jsonl(path) -> list[dict]`

`write_jsonl` writes one compact JSON object per line (UTF-8, `ensure_ascii=False`), serializing `datetime`/`date`/`Decimal` via `default=str`; returns the count. `read_jsonl` reads them back, skipping blank lines.

**Input:** `path, records`

**Expected output:** `int`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_jsonl_roundtrip` in `tests/files/test_06_03_json_config_env.py`.

**Check:** `pytest tests/files/test_06_03_json_config_env.py -k test_jsonl_roundtrip`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Config file + environment overrides with type coercion

**Implement:** `load_config(path, env: dict | None = None, prefix: str = 'LAB_') -> dict`

Load a JSON config (like `datasets/config/pipeline.json`), then override **top-level** keys from environment variables named `PREFIX + KEY.upper()`
(`LAB_BATCH_SIZE=250` overrides `batch_size`). Coerce env strings to the type of the existing value: `int`, `float`, `bool` (`true/false/1/0/yes/no`,
case-insensitive; anything else → `ValueError`), `list` (comma-separated, stripped). `env` defaults to `os.environ`. Env vars for keys not in the file are ignored.

**Input:** `path, env: dict | None = None, prefix: str = 'LAB_'`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_load_config, test_load_config_real_file` in `tests/files/test_06_03_json_config_env.py`.

**Check:** `pytest tests/files/test_06_03_json_config_env.py -k "test_load_config or test_load_config_real_file"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Customer summary from JSON

**Implement:** `summarize_customers(path, output_path) -> dict`

Read `datasets/incoming/customers.json` (a JSON array). Write a pretty-printed (indent 2, sorted keys) JSON summary to `output_path`:
`{"total": n, "active": n, "by_country": {country: count}, "by_segment": {segment: count}, "signup_range": [min_date, max_date]}` where active means
`is_active` is `true` or `"true"`. Return the summary dict. Empty array → zeros, empty dicts, `signup_range` `[null, null]`.

**Input:** `path, output_path`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_summarize_customers` in `tests/files/test_06_03_json_config_env.py`.

**Check:** `pytest tests/files/test_06_03_json_config_env.py -k test_summarize_customers`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Validate nested JSON against a lightweight schema

**Implement:** `validate_json(doc, schema, path: str = '$') -> list[str]`

Schema language: a **type or tuple of types** (`int`, `(int, float)`, `str`); a **dict** (required keys → sub-schemas; a key ending in `?` is optional
and may also be `None`); a **one-element list** `[sub_schema]` (every element must match). Return error strings like
`"$.items[1].price: expected int|float, got str"`, `"$.customer.email: missing"`, `"$.items: expected list, got dict"`. Booleans must not satisfy `int`.

**Input:** `doc, schema, path: str = '$'`

**Expected output:** `list[str]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
schema = {"id": int, "customer": {"name": str, "email?": str}, "items": [{"name": str, "price": (int, float)}]}
validate_json({"id": 1, "customer": {"name": "P"}, "items": [{"name": "Laptop", "price": "70000"}]}, schema)
# ['$.items[0].price: expected int|float, got str']
```

**Check:** `pytest tests/files/test_06_03_json_config_env.py -k "test_validate_json_ok or test_validate_json_errors"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Layered settings with provenance · 🧩 Mini-project

**Implement:** `class Settings`

`Settings(defaults: dict, json_path=None, env_file=None, environ=None, prefix="LAB_", overrides=None, required=())` merges sources with increasing priority:
defaults → JSON file → `.env` file → environment (`environ`, default `os.environ`, only `PREFIX*` vars) → explicit `overrides`.
- `.env` parsing: `KEY=VALUE` lines, `#` comments and blank lines ignored, optional `export ` prefix, surrounding single/double quotes stripped, inline ` # comment` after unquoted values removed.
- Keys from `.env`/environ are normalized: strip the prefix if present, lowercase (`LAB_BATCH_SIZE` → `batch_size`).
- Values from string sources are coerced to the type of the **default** (same rules as before: int/float/bool/list); keys without a default stay strings.
- `settings["key"]`, `settings.get(key, default)`, `"key" in settings`, `settings.source_of("key")` → `"defaults"|"json"|"env_file"|"environ"|"overrides"`, `settings.as_dict()`.
- Missing `required` keys after merging → `ValueError` listing all of them.
Try it with `datasets/config/pipeline.json` and `datasets/config/.env.example`.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
parse_env_file("A=1\nB = ' x # y '\nC=v # c\n#D=4")
# -> {'A': '1', 'B': ' x # y ', 'C': 'v'}
```

**Check:** `pytest tests/files/test_06_03_json_config_env.py -k "test_settings_layers or test_settings_required_and_missing_files or test_parse_env_file"`

**YOUR CODE ↓**